# Privacy-Preserving Record Linkage (PPRL) Demonstration

This notebook is the recommended walkthrough for the `pprl-superhero-example` demo. It shows how two organizations can link records **without exchanging raw identifiers** (like names or Social Security Numbers).

**Scenario:** Super Hero Hospital and Super Hero Pharmacy want to link patient records for care coordination while protecting privacy.

**Who does what:** In this demo, assume the **hospital** runs the overlap analysis step (it receives the pharmacy token file and compares tokens to find matches).

**Match policy:** OpenToken provides standard token rules (T1–T5), but it does **not** define a single, universal match policy. A match policy is what the parties agree on: **which token IDs must match (and how many)** before treating two records as the same person. This notebook shows strict matching (T1–T5) and an example of relaxing the policy.

**Custom tokens (optional):** Parties can define additional token rules beyond T1–T5, as long as both sides use the exact same token definitions and normalization rules (otherwise tokens won’t be comparable).

**Prefer a one-command run?** From this directory you can run `./run_end_to_end.sh` to generate data, tokenize, and analyze overlap end-to-end (see `README.md`).

> **Important demo disclaimer:** This example uses fully synthetic data and example secrets that are safe for illustration only. Do not reuse these keys or patterns in production. Real deployments must use strong key management, strict access controls around decryption and matching, and clear governance over who can run linkage jobs and how results are used.

In [ ]:
# Setup: imports, paths, and dependency availability
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd


def _find_demo_dir() -> Path:
    """Find the demo directory regardless of the current working directory."""
    start = Path.cwd().resolve()
    candidates = [start, *start.parents]
    for base in candidates:
        if (base / "scripts" / "generate_superhero_datasets.py").exists():
            return base
        demo = base / "demos" / "pprl-superhero-example"
        if (demo / "scripts" / "generate_superhero_datasets.py").exists():
            return demo
    raise FileNotFoundError("Could not locate demos/pprl-superhero-example")


demo_dir = _find_demo_dir()
project_root = demo_dir.parent.parent
scripts_dir = demo_dir / "scripts"
datasets_dir = demo_dir / "datasets"
outputs_dir = demo_dir / "outputs"
pyspark_outputs_dir = outputs_dir / "pyspark"
non_pyspark_outputs_dir = outputs_dir / "non_pyspark"

for directory in [datasets_dir, outputs_dir, pyspark_outputs_dir, non_pyspark_outputs_dir]:
    directory.mkdir(parents=True, exist_ok=True)

HASHING_SECRET = "SuperHeroHashingKey2024"
ENCRYPTION_KEY = "SuperHero-Encryption-Key-32chars"
if len(ENCRYPTION_KEY) != 32:
    raise ValueError(f"Encryption key must be exactly 32 characters, got {len(ENCRYPTION_KEY)}")

spark = None
pyspark_available = False
pyspark_import_error = None

try:
    from pyspark.sql import SparkSession  # type: ignore
    from opentoken_pyspark import OpenTokenProcessor  # type: ignore
    from opentoken_pyspark.overlap_analyzer import OpenTokenOverlapAnalyzer  # type: ignore
    pyspark_available = True
except Exception as exc:  # pragma: no cover - notebook helper
    pyspark_import_error = exc

print(f"Demo directory: {demo_dir}")
print(f"PySpark outputs: {pyspark_outputs_dir}")
print(f"Non-PySpark outputs: {non_pyspark_outputs_dir}")
print()

if pyspark_available:
    print("PySpark support is available in this notebook environment.")
    print("- Continue to Section 4 for the PySpark flow.")
    print("- Or jump to Section 5 if you want the non-PySpark flow instead.")
else:
    print("PySpark support is not available in this notebook environment.")
    print("- Run the install cell below, then rerun this setup cell if you want the PySpark flow.")
    print("- Otherwise jump to Section 5 for the non-PySpark flow.")
    print(f"Import status: {type(pyspark_import_error).__name__}: {pyspark_import_error}")

## 1A. Optional: install PySpark support in this notebook

Run the next cell only if you want to follow the **PySpark flow** in Section 4. After it finishes, rerun the setup cell above so the notebook picks up the new dependencies.


In [ ]:
# Optional: install PySpark support for the notebook kernel.
print("$ python -m pip install pyspark")
subprocess.run(["python", "-m", "pip", "install", "pyspark"], cwd=str(demo_dir), check=True)
print()
print("$ python -m pip install -e ../../lib/python/opentoken -e ../../lib/python/opentoken-pyspark")
subprocess.run(
    [
        "python",
        "-m",
        "pip",
        "install",
        "-e",
        "../../lib/python/opentoken",
        "-e",
        "../../lib/python/opentoken-pyspark",
    ],
    cwd=str(demo_dir),
    check=True,
)
print()
print("Re-run the setup cell before continuing to the PySpark flow.")

## 2. Generate Superhero Datasets

Create two datasets (hospital and pharmacy) with a 40% overlap. The overlap represents patients that appear in both datasets.

In [ ]:
# Run the data generation script
result = subprocess.run(
    [sys.executable, str(scripts_dir / "generate_superhero_datasets.py")],
    cwd=str(demo_dir),
    capture_output=True,
    text=True,
    check=False,
 )

print(result.stdout)
if result.returncode != 0:
    print(f"Error: {result.stderr}")
    raise RuntimeError("Dataset generation failed")
print("✓ Datasets generated successfully!")

### Inspect the Generated Datasets

In [ ]:
# Load and display hospital dataset
hospital_df = pd.read_csv(datasets_dir / 'hospital_superhero_data.csv')
print(f"Hospital Dataset: {len(hospital_df)} records")
print(hospital_df.head())
print()

# Load and display pharmacy dataset
pharmacy_df = pd.read_csv(datasets_dir / 'pharmacy_superhero_data.csv')
print(f"Pharmacy Dataset: {len(pharmacy_df)} records")
print(pharmacy_df.head())

## 3. Choose an Execution Path

This notebook now presents the two implementations as separate guided flows.

- If the setup cell says PySpark is available, continue to **Section 4. PySpark Flow**.
- If PySpark is unavailable, or you want the script-driven demo, jump to **Section 5. Non-PySpark Flow**.
- In either section, the main path demonstrates **strict matching** first and then offers an optional cell to try **relaxed matching**.


## 4. PySpark Flow

This flow shows the in-notebook PySpark implementation directly. Run the cells in order to tokenize the datasets, inspect the tokenized output, run the strict match policy, inspect one matched pair, and then optionally try a relaxed match policy.


### Tokenize the datasets with the PySpark bridge

This cell creates a local Spark session, loads each dataset, tokenizes it with the OpenToken PySpark bridge, and writes the token CSV files into `outputs/pyspark/`.


In [ ]:
# Tokenize both datasets with the OpenToken PySpark bridge.
if not pyspark_available:
    raise RuntimeError(
        "PySpark is not available. Run the install cell and rerun setup, or jump to Section 5 instead."
    )

spark = (
    SparkSession.builder
    .appName("PPRL-Superhero-Demo")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

processor = OpenTokenProcessor(
    hashing_secret=HASHING_SECRET,
    encryption_key=ENCRYPTION_KEY,
)

hospital_spark_df = spark.read.csv(
    str(datasets_dir / "hospital_superhero_data.csv"),
    header=True,
    inferSchema=True,
)
pharmacy_spark_df = spark.read.csv(
    str(datasets_dir / "pharmacy_superhero_data.csv"),
    header=True,
    inferSchema=True,
)

print(f"Hospital input rows: {hospital_spark_df.count()}")
print(f"Pharmacy input rows: {pharmacy_spark_df.count()}")
print()
print("Generating encrypted tokens with the OpenToken PySpark bridge...")

hospital_tokens_spark = processor.process_dataframe(hospital_spark_df)
pharmacy_tokens_spark = processor.process_dataframe(pharmacy_spark_df)

hospital_temp_dir = pyspark_outputs_dir / "hospital_tokens_temp"
pharmacy_temp_dir = pyspark_outputs_dir / "pharmacy_tokens_temp"
for temp_dir in [hospital_temp_dir, pharmacy_temp_dir]:
    if temp_dir.exists():
        shutil.rmtree(temp_dir)

hospital_tokens_spark.coalesce(1).write.mode("overwrite").option("header", "true").csv(str(hospital_temp_dir))
pharmacy_tokens_spark.coalesce(1).write.mode("overwrite").option("header", "true").csv(str(pharmacy_temp_dir))

hospital_csv = sorted(hospital_temp_dir.glob("*.csv"))[0]
pharmacy_csv = sorted(pharmacy_temp_dir.glob("*.csv"))[0]
shutil.copy2(hospital_csv, pyspark_outputs_dir / "hospital_tokens.csv")
shutil.copy2(pharmacy_csv, pyspark_outputs_dir / "pharmacy_tokens.csv")
shutil.rmtree(hospital_temp_dir)
shutil.rmtree(pharmacy_temp_dir)

print(f"Saved hospital tokens to {pyspark_outputs_dir / 'hospital_tokens.csv'}")
print(f"Saved pharmacy tokens to {pyspark_outputs_dir / 'pharmacy_tokens.csv'}")
print(f"Hospital token rows: {hospital_tokens_spark.count()}")
print(f"Pharmacy token rows: {pharmacy_tokens_spark.count()}")

### Inspect the PySpark tokenized output

This cell loads the saved CSV artifacts for the PySpark flow so you can see the token rows that were just produced.


In [ ]:
# Inspect the token files produced by the PySpark flow.
pyspark_hospital_tokens = pd.read_csv(pyspark_outputs_dir / "hospital_tokens.csv")
pyspark_pharmacy_tokens = pd.read_csv(pyspark_outputs_dir / "pharmacy_tokens.csv")

print(f"PySpark hospital token rows: {len(pyspark_hospital_tokens)}")
print(pyspark_hospital_tokens.head(10))
print()
print(f"PySpark pharmacy token rows: {len(pyspark_pharmacy_tokens)}")
print(pyspark_pharmacy_tokens.head(10))

### Run the strict PySpark match policy

This cell decrypts the PySpark token files inside the analyzer, requires all five standard token rules (`T1`-`T5`) to match, and saves the resulting linked record pairs to `outputs/pyspark/matching_records.csv`.


In [ ]:
# Run the strict PySpark overlap analysis and save the linked pairs.
if spark is None:
    raise RuntimeError("Run the PySpark tokenization cell before the strict overlap cell.")

from pyspark.sql.functions import lit  # type: ignore

hospital_tokens_spark = spark.read.csv(
    str(pyspark_outputs_dir / "hospital_tokens.csv"),
    header=True,
    inferSchema=True,
)
pharmacy_tokens_spark = spark.read.csv(
    str(pyspark_outputs_dir / "pharmacy_tokens.csv"),
    header=True,
    inferSchema=True,
)

strict_rules = ["T1", "T2", "T3", "T4", "T5"]
analyzer = OpenTokenOverlapAnalyzer(ENCRYPTION_KEY)
results = analyzer.analyze_overlap(
    hospital_tokens_spark,
    pharmacy_tokens_spark,
    matching_rules=strict_rules,
    dataset1_name="Hospital",
    dataset2_name="Pharmacy",
)

print("PySpark strict overlap summary:")
print(f"  Hospital records: {results['total_records_dataset1']}")
print(f"  Pharmacy records: {results['total_records_dataset2']}")
print(f"  Matching hospital records: {results['matching_records_dataset1']}")
print(f"  Matching pharmacy records: {results['matching_records_dataset2']}")
print(f"  Overlap percentage: {results['overlap_percentage']:.1f}%")

pyspark_matches_spark = results["matches"].withColumn("MatchingTokens", lit("|".join(strict_rules))).withColumn(
    "TokenCount",
    lit(len(strict_rules)),
)

strict_temp_dir = pyspark_outputs_dir / "matching_records_temp"
if strict_temp_dir.exists():
    shutil.rmtree(strict_temp_dir)
pyspark_matches_spark.coalesce(1).write.mode("overwrite").option("header", "true").csv(str(strict_temp_dir))
strict_csv = sorted(strict_temp_dir.glob("*.csv"))[0]
shutil.copy2(strict_csv, pyspark_outputs_dir / "matching_records.csv")
shutil.rmtree(strict_temp_dir)

pyspark_matches_df = pd.read_csv(pyspark_outputs_dir / "matching_records.csv")
print()
print(pyspark_matches_df.head(10))

### Inspect one strict PySpark match

This cell uses the strict-match output from the PySpark flow and looks up the corresponding raw records so you can see what one linked pair represents.


In [ ]:
# Inspect one strict match produced by the PySpark flow.
pyspark_matches_df = pd.read_csv(pyspark_outputs_dir / "matching_records.csv")

if len(pyspark_matches_df) == 0:
    print("No strict PySpark matches are available yet.")
else:
    sample_match = pyspark_matches_df.iloc[0]
    hospital_record_id = sample_match["Hospital_RecordId"]
    pharmacy_record_id = sample_match["Pharmacy_RecordId"]

    hospital_match = hospital_df[hospital_df["RecordId"] == hospital_record_id]
    pharmacy_match = pharmacy_df[pharmacy_df["RecordId"] == pharmacy_record_id]
    hospital_record = hospital_match.iloc[0]
    pharmacy_record = pharmacy_match.iloc[0]

    print(f"Hospital Record ID: {hospital_record_id}")
    print(f"Hospital Patient: {hospital_record['FirstName']} {hospital_record['LastName']}")
    print(f"DOB: {hospital_record['BirthDate']}, SSN: {hospital_record['SocialSecurityNumber']}")
    print()
    print(f"Pharmacy Record ID: {pharmacy_record_id}")
    print(f"Pharmacy Patient: {pharmacy_record['FirstName']} {pharmacy_record['LastName']}")
    print(f"DOB: {pharmacy_record['BirthDate']}, SSN: {pharmacy_record['SocialSecurityNumber']}")
    print()
    print("✓ The strict PySpark flow linked the same synthetic patient across both datasets.")

### Optional: try a relaxed PySpark match policy

If you want to see how the results change when the match policy is loosened, run this cell. It repeats the overlap analysis but requires only `T1`, `T2`, `T3`, and `T5` to match.


In [ ]:
# Re-run the PySpark overlap analysis with a relaxed match policy.
if spark is None:
    raise RuntimeError("Run the PySpark tokenization cell before the relaxed overlap cell.")

from pyspark.sql.functions import lit  # type: ignore

hospital_tokens_spark = spark.read.csv(
    str(pyspark_outputs_dir / "hospital_tokens.csv"),
    header=True,
    inferSchema=True,
)
pharmacy_tokens_spark = spark.read.csv(
    str(pyspark_outputs_dir / "pharmacy_tokens.csv"),
    header=True,
    inferSchema=True,
)

relaxed_rules = ["T1", "T2", "T3", "T5"]
analyzer = OpenTokenOverlapAnalyzer(ENCRYPTION_KEY)
results_relaxed = analyzer.analyze_overlap(
    hospital_tokens_spark,
    pharmacy_tokens_spark,
    matching_rules=relaxed_rules,
    dataset1_name="Hospital",
    dataset2_name="Pharmacy",
)

print("PySpark relaxed overlap summary:")
print(f"  Matching hospital records: {results_relaxed['matching_records_dataset1']}")
print(f"  Matching pharmacy records: {results_relaxed['matching_records_dataset2']}")
print(f"  Overlap percentage: {results_relaxed['overlap_percentage']:.1f}%")

pyspark_relaxed_matches_spark = results_relaxed["matches"].withColumn(
    "MatchingTokens",
    lit("|".join(relaxed_rules)),
).withColumn("TokenCount", lit(len(relaxed_rules)))

relaxed_temp_dir = pyspark_outputs_dir / "matching_records_alt_temp"
if relaxed_temp_dir.exists():
    shutil.rmtree(relaxed_temp_dir)
pyspark_relaxed_matches_spark.coalesce(1).write.mode("overwrite").option("header", "true").csv(str(relaxed_temp_dir))
relaxed_csv = sorted(relaxed_temp_dir.glob("*.csv"))[0]
shutil.copy2(relaxed_csv, pyspark_outputs_dir / "matching_records_alt.csv")
shutil.rmtree(relaxed_temp_dir)

pyspark_relaxed_matches_df = pd.read_csv(pyspark_outputs_dir / "matching_records_alt.csv")
print()
print(pyspark_relaxed_matches_df.head(10))

## 5. Non-PySpark Flow

This flow shows the script-driven implementation directly. Run the cells in order to tokenize the datasets, inspect the tokenized output, run the strict match policy, inspect one matched pair, and then optionally try a relaxed match policy.


### Tokenize the datasets with the demo scripts

This cell runs the existing shell scripts that call the Java CLI, then copies the generated CSV artifacts into `outputs/non_pyspark/` so this flow can be reviewed independently.


In [ ]:
# Tokenize both datasets with the non-PySpark demo scripts.
print("$ bash scripts/tokenize_hospital.sh")
hospital_result = subprocess.run(
    ["bash", str(scripts_dir / "tokenize_hospital.sh")],
    cwd=str(demo_dir),
    capture_output=True,
    text=True,
    check=False,
)
print(hospital_result.stdout)
if hospital_result.returncode != 0:
    print(hospital_result.stderr)
    raise RuntimeError("Hospital tokenization failed")

print("$ bash scripts/tokenize_pharmacy.sh")
pharmacy_result = subprocess.run(
    ["bash", str(scripts_dir / "tokenize_pharmacy.sh")],
    cwd=str(demo_dir),
    capture_output=True,
    text=True,
    check=False,
)
print(pharmacy_result.stdout)
if pharmacy_result.returncode != 0:
    print(pharmacy_result.stderr)
    raise RuntimeError("Pharmacy tokenization failed")

for filename in [
    "hospital_tokens.csv",
    "hospital_tokens.metadata.json",
    "hospital_tokens.csv.metadata.json",
    "pharmacy_tokens.csv",
    "pharmacy_tokens.metadata.json",
    "pharmacy_tokens.csv.metadata.json",
]:
    source = outputs_dir / filename
    if source.exists():
        shutil.copy2(source, non_pyspark_outputs_dir / filename)

print(f"Saved non-PySpark artifacts under {non_pyspark_outputs_dir}")

### Inspect the non-PySpark tokenized output

This cell loads the saved CSV artifacts for the non-PySpark flow so you can see the token rows produced by the shell-script path.


In [ ]:
# Inspect the token files produced by the non-PySpark flow.
non_pyspark_hospital_tokens = pd.read_csv(non_pyspark_outputs_dir / "hospital_tokens.csv")
non_pyspark_pharmacy_tokens = pd.read_csv(non_pyspark_outputs_dir / "pharmacy_tokens.csv")

print(f"Non-PySpark hospital token rows: {len(non_pyspark_hospital_tokens)}")
print(non_pyspark_hospital_tokens.head(10))
print()
print(f"Non-PySpark pharmacy token rows: {len(non_pyspark_pharmacy_tokens)}")
print(non_pyspark_pharmacy_tokens.head(10))

### Run the strict non-PySpark match policy

This cell runs the overlap-analysis script directly, requires all five standard token rules (`T1`-`T5`) to match, and saves the linked record pairs to `outputs/non_pyspark/matching_records.csv`.


In [ ]:
# Run the strict non-PySpark overlap analysis and save the linked pairs.
print("$ python scripts/analyze_overlap.py --output matching_records.csv")
strict_result = subprocess.run(
    [sys.executable, str(scripts_dir / "analyze_overlap.py"), "--output", "matching_records.csv"],
    cwd=str(demo_dir),
    capture_output=True,
    text=True,
    check=False,
)
print(strict_result.stdout)
if strict_result.returncode != 0:
    print(strict_result.stderr)
    raise RuntimeError("Non-PySpark strict overlap analysis failed")

shutil.copy2(outputs_dir / "matching_records.csv", non_pyspark_outputs_dir / "matching_records.csv")
non_pyspark_matches_df = pd.read_csv(non_pyspark_outputs_dir / "matching_records.csv")
print(non_pyspark_matches_df.head(10))

### Inspect one strict non-PySpark match

This cell uses the strict-match output from the non-PySpark flow and looks up the corresponding raw records so you can see what one linked pair represents.


In [ ]:
# Inspect one strict match produced by the non-PySpark flow.
non_pyspark_matches_df = pd.read_csv(non_pyspark_outputs_dir / "matching_records.csv")

if len(non_pyspark_matches_df) == 0:
    print("No strict non-PySpark matches are available yet.")
else:
    sample_match = non_pyspark_matches_df.iloc[0]
    hospital_record_id = sample_match["Hospital_RecordId"]
    pharmacy_record_id = sample_match["Pharmacy_RecordId"]

    hospital_match = hospital_df[hospital_df["RecordId"] == hospital_record_id]
    pharmacy_match = pharmacy_df[pharmacy_df["RecordId"] == pharmacy_record_id]
    hospital_record = hospital_match.iloc[0]
    pharmacy_record = pharmacy_match.iloc[0]

    print(f"Hospital Record ID: {hospital_record_id}")
    print(f"Hospital Patient: {hospital_record['FirstName']} {hospital_record['LastName']}")
    print(f"DOB: {hospital_record['BirthDate']}, SSN: {hospital_record['SocialSecurityNumber']}")
    print()
    print(f"Pharmacy Record ID: {pharmacy_record_id}")
    print(f"Pharmacy Patient: {pharmacy_record['FirstName']} {pharmacy_record['LastName']}")
    print(f"DOB: {pharmacy_record['BirthDate']}, SSN: {pharmacy_record['SocialSecurityNumber']}")
    print()
    print("✓ The strict non-PySpark flow linked the same synthetic patient across both datasets.")

### Optional: try a relaxed non-PySpark match policy

If you want to see how the results change when the match policy is loosened, run this cell. It repeats the overlap analysis but requires only `T1`, `T2`, `T3`, and `T5` to match.


In [ ]:
# Re-run the non-PySpark overlap analysis with a relaxed match policy.
print("$ python scripts/analyze_overlap.py --matching-rules T1 T2 T3 T5 --output matching_records_alt.csv")
relaxed_result = subprocess.run(
    [
        sys.executable,
        str(scripts_dir / "analyze_overlap.py"),
        "--matching-rules",
        "T1",
        "T2",
        "T3",
        "T5",
        "--output",
        "matching_records_alt.csv",
    ],
    cwd=str(demo_dir),
    capture_output=True,
    text=True,
    check=False,
)
print(relaxed_result.stdout)
if relaxed_result.returncode != 0:
    print(relaxed_result.stderr)
    raise RuntimeError("Non-PySpark relaxed overlap analysis failed")

shutil.copy2(outputs_dir / "matching_records_alt.csv", non_pyspark_outputs_dir / "matching_records_alt.csv")
non_pyspark_relaxed_matches_df = pd.read_csv(non_pyspark_outputs_dir / "matching_records_alt.csv")
print(non_pyspark_relaxed_matches_df.head(10))

## 8. Privacy and Security Summary

This demonstration shows how OpenToken enables privacy-preserving record linkage:

### What was protected:
- ✓ Raw patient data (names, SSNs, birthdates) was never shared between organizations
- ✓ HMAC-SHA256 hashes cannot be reversed to recover original data
- ✓ Encryption key controls who can decrypt and perform linkage

### What was shared:
- • Encrypted tokens for secure transmission
- • Matching statistics showing overlap counts
- • Metadata with summary information (not raw data)

### Key security principles:
1. **Strong Encryption**: AES-256-GCM with random IVs prevents pattern analysis
2. **Key Management**: Secure sharing and storage of encryption/hashing keys
3. **Deterministic Hashing**: HMAC-SHA256 enables comparison without raw data
4. **Access Control**: Only authorized parties can decrypt tokens

### PySpark Bridge Benefits:
- **Distributed Processing**: Handle large datasets across multiple nodes
- **Parallel Decryption**: Efficiently decrypt millions of tokens
- **Scalable Analysis**: Perform overlap analysis on enterprise-scale data
- **Integration**: Native Spark SQL for additional transformations

## 9. Customization Examples

You can customize this demo by modifying the scripts:

### Change dataset size and overlap:
Edit `scripts/generate_superhero_datasets.py`:
```python
num_hospital = 500  # Different size
num_pharmacy = 600
overlap_percentage = 0.50  # 50% overlap instead of 40%
```

### Use different secrets (demo only):
Edit **both** tokenization scripts to keep them in sync:
- `scripts/tokenize_hospital.sh`
- `scripts/tokenize_pharmacy.sh`

```bash
HASHING_SECRET="YourCustomHashingKey"
ENCRYPTION_KEY="YourCustomEncryptionKey-32chars"  # Must be exactly 32 characters
```

**Important**: Both organizations must use the same secrets for tokens to match.

### Scale with PySpark:
For large datasets, ensure PySpark and the OpenToken PySpark bridge are installed:
```bash
pip install pyspark opentoken-pyspark
```

The notebook will automatically use distributed processing if available.

## 10. Next Steps

This PPRL demo can be adapted for:
- Healthcare: Hospital-to-hospital patient matching
- Insurance: Claims linkage across providers
- Research: Multi-site study participant matching
- Government: Cross-agency identity resolution
- Financial Services: Anti-fraud systems

### With PySpark Bridge:
- Scale to petabyte-level datasets
- Distribute tokenization across clusters
- Parallel overlap analysis
- Real-time record linkage pipelines

For more information, see the [README.md](./README.md) in this directory and the [main OpenToken documentation](../../README.md).

<!-- notebook-edit-test -->